# Recuperar Clientes

Identifica clientes que no han comprado en un rango de días configurable (por defecto 25-366 días), para generar un listado de campañas de recuperación.

**Fixes aplicados respecto a la versión original:**
- Se protege contra división por hora del día (`fecha_actual` normalizada a medianoche).
- `exit()` reemplazado por control de flujo `if/else` (más seguro en notebooks).
- Nombres de cliente normalizados (`strip` + `upper`) para evitar duplicados por espacios/mayúsculas.
- Validación explícita de columnas requeridas, con mensaje de error claro.
- El montaje de Google Drive solo se ejecuta si el notebook corre en Colab (funciona también en local).
- Umbrales de días configurables como constantes al inicio del script.


In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import calendar

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
ARCHIVO = "/content/drive/MyDrive/Sales BILANNATURALS 2023-Mar2026.xlsx"
NOMBRE_HOJA = "Reporte Ventas"
DIAS_MIN_INACTIVIDAD = 25
DIAS_MAX_INACTIVIDAD = 366
COLUMNAS_REQUERIDAS = ["fecha", "cliente"]

# Intentar montar Google Drive solo si estamos en Colab
try:
    from google.colab import drive, files
    drive.mount('/content/drive', force_remount=False)
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
    print("Ejecutando fuera de Google Colab: se omite el montaje de Drive.")


In [ ]:
# ---------------------------------------------------------
# FUNCIONES
# ---------------------------------------------------------

def es_ultimo_dia_habil(fecha):
    """
    Retorna True si la fecha es el último día hábil del mes.
    Incluye sábado como día hábil; si el último día cae domingo, retrocede a sábado.
    """
    anio, mes = fecha.year, fecha.month
    ultimo_dia_mes = calendar.monthrange(anio, mes)[1]
    fecha_ultimo = datetime(anio, mes, ultimo_dia_mes)

    if fecha_ultimo.weekday() == 6:  # domingo
        fecha_ultimo -= timedelta(days=1)

    return fecha.date() == fecha_ultimo.date()


def cargar_datos(archivo, hoja, columnas_requeridas):
    """Carga y limpia el Excel de ventas, validando columnas requeridas."""
    df = pd.read_excel(archivo, sheet_name=hoja)
    df.columns = df.columns.str.strip().str.lower()

    faltantes = [c for c in columnas_requeridas if c not in df.columns]
    if faltantes:
        raise ValueError(
            f"Faltan columnas requeridas en el Excel: {faltantes}. "
            f"Columnas encontradas: {list(df.columns)}"
        )

    df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True, errors="coerce")
    df["cliente"] = df["cliente"].astype(str).str.strip().str.upper()

    filas_antes = len(df)
    df = df.dropna(subset=["fecha", "cliente"])
    filas_descartadas = filas_antes - len(df)
    if filas_descartadas > 0:
        print(f"Se descartaron {filas_descartadas} filas por fecha o cliente inválido.")

    return df


def calcular_clientes_recuperacion(df, dias_min, dias_max):
    """Calcula última compra y días de inactividad por cliente, y filtra por rango."""
    ultima_compra = (
        df.groupby("cliente")["fecha"]
        .max()
        .reset_index()
        .rename(columns={"fecha": "ultima_fecha_compra"})
    )

    # Normalizada a medianoche para que el delta no dependa de la hora de ejecución
    fecha_actual = pd.Timestamp(datetime.now().date())
    ultima_compra["delta_dias"] = (fecha_actual - ultima_compra["ultima_fecha_compra"]).dt.days

    clientes_recompra = ultima_compra[
        (ultima_compra["delta_dias"] >= dias_min) &
        (ultima_compra["delta_dias"] <= dias_max)
    ].copy()

    return clientes_recompra, fecha_actual


def generar_nombre_archivo(fecha_actual, dias_max, dias_min):
    """Genera el nombre del Excel de salida con el rango de fechas evaluado."""
    fecha_inicio = (fecha_actual - timedelta(days=dias_max)).strftime("%d %b %Y").lower()
    fecha_fin = (fecha_actual - timedelta(days=dias_min)).strftime("%d %b %Y").lower()
    nombre = f"Clientes {dias_min}-{dias_max} dias sin comprar {fecha_inicio} - {fecha_fin}.xlsx"

    reemplazos = {"á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u"}
    for acento, plano in reemplazos.items():
        nombre = nombre.replace(acento, plano)
    return nombre


In [ ]:
# ---------------------------------------------------------
# EJECUCIÓN
# ---------------------------------------------------------
df = cargar_datos(ARCHIVO, NOMBRE_HOJA, COLUMNAS_REQUERIDAS)
clientes_recompra, fecha_actual = calcular_clientes_recuperacion(
    df, DIAS_MIN_INACTIVIDAD, DIAS_MAX_INACTIVIDAD
)

if clientes_recompra.empty:
    print(f"No hay clientes con inactividad entre {DIAS_MIN_INACTIVIDAD} y {DIAS_MAX_INACTIVIDAD} días.")
else:
    nombre_archivo = generar_nombre_archivo(fecha_actual, DIAS_MAX_INACTIVIDAD, DIAS_MIN_INACTIVIDAD)
    clientes_recompra.to_excel(nombre_archivo, index=False)

    print(f"Archivo generado correctamente: {nombre_archivo}")
    print(f"Clientes exportados: {len(clientes_recompra)}")

    if es_ultimo_dia_habil(fecha_actual):
        print("Hoy es el último día hábil del mes.")
    else:
        print("Proceso ejecutado en modo diario.")

    if EN_COLAB:
        files.download(nombre_archivo)
